In [ ]:
from google.colab import drive

print('All imports OK')
drive.mount('/content/drive', force_remount=True)


In [ ]:
# ── 0. Install ────────────────────────────────────────────────────────────────
!pip install librosa torch torchaudio torchvggish -q

In [ ]:
# ── 2. Load labels (reuse your existing parser) ───────────────────────────────
def split_phones(text):
    phones, i = [], 0
    while i < len(text):
        if i+1 < len(text) and text[i+1] == "ː":
            phones.append(text[i:i+2]); i += 2
        elif text[i] != " ":
            phones.append(text[i]); i += 1
        else:
            i += 1
    return " ".join(phones)

def load_jsonl(path):
    mapping = {}
    with open(path) as f:
        for line in f:
            r = json.loads(line)
            mapping[r["utterance_id"] + ".flac"] = split_phones(r["phonetic_text"])
    return mapping

labels_dict = load_jsonl(LABELS_JSON)

all_phones  = {p for v in labels_dict.values() for p in v.split()}
vocab       = ['<blank>'] + sorted(all_phones)
phone2idx   = {p: i for i, p in enumerate(vocab)}
idx2phone   = {i: p for p, i in phone2idx.items()}
NUM_CLASSES = len(vocab)
print(f"Vocab: {NUM_CLASSES} | Device: {DEVICE}")

# ── 3. Train/val split ────────────────────────────────────────────────────────
items = list(labels_dict.items())
random.shuffle(items)
split      = int(0.9 * len(items))
train_dict = dict(items[:split])
val_dict   = dict(items[split:])


In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# BENCHMARK A — Plain BiLSTM on MFCCs
# ══════════════════════════════════════════════════════════════════════════════

def safe_delta(mfcc):
    w = min(9, mfcc.shape[1])
    if w % 2 == 0: w -= 1
    return librosa.feature.delta(mfcc, width=max(w, 3))

class MFCCDataset(Dataset):
    def __init__(self, d):
        self.items = list(d.items())
    def __len__(self): return len(self.items)
    def __getitem__(self, idx):
        fname, phone_str = self.items[idx]
        y, _ = librosa.load(AUDIO_DIR / fname, sr=SAMPLE_RATE)
        mfcc  = librosa.feature.mfcc(y=y, sr=SAMPLE_RATE, n_mfcc=N_MFCC)
        feat  = np.vstack([mfcc, safe_delta(mfcc), safe_delta(safe_delta(mfcc))]).T
        feat  = (feat - feat.mean(0)) / (feat.std(0) + 1e-8)
        label = torch.LongTensor([phone2idx[p] for p in phone_str.split() if p in phone2idx])
        return torch.FloatTensor(feat), label

def collate(batch):
    feats, labels = zip(*batch)
    return (pad_sequence(feats, batch_first=True),
            torch.cat(labels),
            torch.LongTensor([f.shape[0] for f in feats]),
            torch.LongTensor([l.shape[0] for l in labels]))

train_dl = DataLoader(MFCCDataset(train_dict), BATCH_SIZE, shuffle=True,  collate_fn=collate)
val_dl   = DataLoader(MFCCDataset(val_dict),   BATCH_SIZE, shuffle=False, collate_fn=collate)

In [ ]:

# ══════════════════════════════════════════════════════════════════════════════
# BENCHMARK B — VGGish pretrained CNN + BiLSTM
# ══════════════════════════════════════════════════════════════════════════════
# VGGish expects: raw waveform at 16kHz → outputs (N, 128) embeddings per 0.96s window

import torchvggish  # pip install torchvggish

class VGGishDataset(Dataset):
    """Returns raw waveform — VGGish does its own internal mel-spectrogram."""
    def __init__(self, d):
        self.items = list(d.items())
    def __len__(self): return len(self.items)
    def __getitem__(self, idx):
        fname, phone_str = self.items[idx]
        y, _ = librosa.load(AUDIO_DIR / fname, sr=SAMPLE_RATE)
        wav   = torch.FloatTensor(y)
        label = torch.LongTensor([phone2idx[p] for p in phone_str.split() if p in phone2idx])
        return wav, label

def collate_vggish(batch):
    wavs, labels = zip(*batch)
    wav_lens   = torch.LongTensor([w.shape[0] for w in wavs])
    label_lens = torch.LongTensor([l.shape[0] for l in labels])
    wavs_pad   = pad_sequence(wavs, batch_first=True)
    return wavs_pad, torch.cat(labels), wav_lens, label_lens

vtrain_dl = DataLoader(VGGishDataset(train_dict), BATCH_SIZE, shuffle=True,  collate_fn=collate_vggish)
vval_dl   = DataLoader(VGGishDataset(val_dict),   BATCH_SIZE, shuffle=False, collate_fn=collate_vggish)

class VGGishBiLSTM(nn.Module):
    def __init__(self, n_cls=NUM_CLASSES, hidden=256, lstm_layers=2):
        super().__init__()
        self.vggish = torchvggish.vggish()
        self.vggish.eval()
        # Freeze VGGish — only train BiLSTM head
        for p in self.vggish.parameters():
            p.requires_grad = False

        # VGGish outputs 128-dim embeddings per ~0.96s frame
        self.lstm = nn.LSTM(128, hidden, lstm_layers, batch_first=True,
                            bidirectional=True, dropout=0.3)
        self.fc   = nn.Linear(hidden*2, n_cls)

    def forward(self, wavs):
        # wavs: (B, T_samples) — process each item in batch
        embeddings = []
        for wav in wavs:
            # VGGish expects numpy float32 at 16kHz
            emb = self.vggish.forward(wav.unsqueeze(0))  # (n_frames, 128)
            embeddings.append(emb)
        # Pad to same number of frames across batch
        x = pad_sequence(embeddings, batch_first=True)   # (B, max_frames, 128)
        x, _ = self.lstm(x)
        return self.fc(x)                                 # (B, max_frames, n_cls)


In [ ]:

# NOTE: VGGish frames (~0.96s each) are coarser than MFCC frames.
# feat_lengths for CTC must reflect number of VGGish output frames, not raw samples.
# We compute this separately in the training loop below.

def train_vggish_model(model, train_dl, val_dl):
    ctc  = nn.CTCLoss(blank=0, zero_infinity=True)
    # Only optimize unfrozen params (LSTM + FC)
    opt  = torch.optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=LR)
    sch  = torch.optim.lr_scheduler.ReduceLROnPlateau(opt, patience=3, factor=0.5)
    best = float('inf')

    VGGISH_FRAME_SAMPLES = int(0.96 * SAMPLE_RATE)  # ~15360 samples per frame

    for ep in range(1, EPOCHS+1):
        model.train(); model.vggish.eval()  # keep VGGish in eval always
        tl = 0
        for wavs, labels, wav_lens, label_lens in train_dl:
            wavs   = wavs.to(DEVICE); labels = labels.to(DEVICE)
            logits = model(wavs)                                # (B, frames, C)
            # Compute output frame lengths from input wav lengths
            feat_lens = (wav_lens // VGGISH_FRAME_SAMPLES).clamp(min=1)
            lp = logits.log_softmax(-1).permute(1,0,2)
            loss = ctc(lp, labels, feat_lens, label_lens)
            opt.zero_grad(); loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 5.0)
            opt.step(); tl += loss.item()

        model.eval(); vl = 0
        with torch.no_grad():
            for wavs, labels, wav_lens, label_lens in vval_dl:
                wavs   = wavs.to(DEVICE); labels = labels.to(DEVICE)
                logits = model(wavs)
                feat_lens = (wav_lens // VGGISH_FRAME_SAMPLES).clamp(min=1)
                lp = logits.log_softmax(-1).permute(1,0,2)
                vl += ctc(lp, labels, feat_lens, label_lens).item()

        tl /= len(train_dl); vl /= len(val_dl)
        sch.step(vl)
        print(f"[vggish] Ep {ep:02d} | Train {tl:.4f} | Val {vl:.4f}")
        if vl < best:
            best = vl
            torch.save(model.state_dict(), 'best_vggish.pt')
            print(f"  ✓ saved (val={vl:.4f})")
    return best

vggish_model = VGGishBiLSTM().to(DEVICE)
best_vggish  = train_vggish_model(vggish_model, vtrain_dl, vval_dl)

# ── Final comparison ──────────────────────────────────────────────────────────
print("\n═══ Benchmark Summary ═══")
print(f"VGGish + BiLSTM         best val loss: {best_vggish:.4f}")
